# Обработка данных

In [1]:
from src.data_processing.loader import DocumentLoader

loader = DocumentLoader("./data/")
docs = loader.load_all_documents()

/home/pc-01/rag_spb/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from src.vector_store import VectorStore

vs = VectorStore()
vs.clear()

print(f"num chunks: {vs.count}")

num chunks: 0


In [3]:
vs.index_all_documents(loader.load_all_documents())

print(f"num chunks after load: {vs.count}")

num chunks after load: 50


# SGR RAG

In [4]:
from src.pipeline import RAGPipeline, RAGResponse

In [5]:
pipe = RAGPipeline(vector_store=vs)
user_query = "Мне необходимо составить инструкцию по получению соц обслуживания для жителя блокадного Ленинграда"

In [6]:
"""
Process a user query through the full SGR RAG pipeline.

SGR Pattern with iterative retrieval:
1. Search: Retrieve relevant context
2. Check relevance: Evaluate each document with LLM
3. Check completeness: Ask LLM if info is sufficient
4. Iterate: For each clarifying question, retrieve and answer
5. Generate: Create final response
6. Response: Return structured answer

Args:
    user_query: User's query/request.

Returns:
    RAGResponse with generated document and sources.
"""
print(f"Processing query with SGR pattern: {user_query}")

# Step 1: Initial Search (Retrieval)
retrieved_results = pipe._retriever.retrieve_formatted(user_query)
print(f"Initial retrieval: {len(retrieved_results)} results")

# Step 2: Check relevance of each document using LLM
relevant_results = pipe._filter_by_relevance(user_query, retrieved_results)
print(f"After relevance check: {len(relevant_results)} results")

if not relevant_results:
    response = pipe._generate_no_info_response(user_query)
    OUTPUT = RAGResponse(
        query=user_query,
        generated_document=response,
        retrieved_results=retrieved_results,
        sources=[],
    )
    print("No relevant documents found")
else:
    # Step 3: Check if information is complete
    additional_questions = pipe._check_information_completeness(
        query=user_query, 
        contexts="\n\n".join(relevant_results),
    )

    # Step 4: Iterative retrieval for additional questions
    additional_qa_pairs = []
    if additional_questions:
        print(f"Information incomplete. Additional questions:", additional_questions)
        additional_qa_pairs = pipe._iterative_retrieval(
            additional_questions=additional_questions, 
            existing_results=relevant_results,
        )
        for pair in additional_qa_pairs:
            print(pair)

    # Step 5: Generate final answer
    final_answer = pipe._generate_final_answer(
        user_query, relevant_results, additional_qa_pairs
    )

    # Step 6: Response
    OUTPUT = RAGResponse(
        query=user_query,
        generated_document=final_answer,
        retrieved_results=relevant_results,
        additional_questions=additional_questions,
        sources=relevant_results,
    )

    print(f"Query processed successfully.")

Processing query with SGR pattern: Мне необходимо составить инструкцию по получению соц обслуживания для жителя блокадного Ленинграда
Initial retrieval: 2 results
After relevance check: 2 results
Information incomplete. Additional questions: ['Какие документы необходимо предоставить для оформления социального обслуживания жителю блокадного Ленинграда?', 'Какие критерии нуждаемости в социальном обслуживании применяются для жителей блокадного Ленинграда?', 'Какие дополнительные льготы (помимо социального обслуживания и транспортных услуг) предусмотрены для жителей блокадного Ленинграда в Санкт-Петербурге?', 'Какие шаги необходимо предпринять для получения путёвки в оздоровительный пансионат?', 'Какие медицинские учреждения или организации предоставляют услуги по социальному обслуживанию жителям блокадного Ленинграда?']
{'question': 'Какие документы необходимо предоставить для оформления социального обслуживания жителю блокадного Ленинграда?', 'answer': '**Перечень документов, необходимых

In [7]:
print(OUTPUT.generated_document)

### **Инструкция по получению социального обслуживания для жителя блокадного Ленинграда**

---

#### **1. Общие положения**
Социальное обслуживание предоставляется жителям блокадного Ленинграда, утратившим способность к самообслуживанию и передвижению, а также нуждающимся в постоянной или временной поддержке. Обслуживание осуществляется в следующих формах:
- **на дому** (социальный работник или сиделка);
- **в полустационарной форме** (социально-бытовая, медицинская и культурная помощь);
- **в стационарной форме** (круглосуточный уход и проживание).

**Особенность:** Жители блокадного Ленинграда принимаются на **стационарное обслуживание во внеочередном порядке**.

---

#### **2. Критерии нуждаемости**
Гражданин признаётся нуждающимся в социальном обслуживании при наличии **одного или нескольких** следующих условий:
- **полная или частичная утрата возможности самостоятельного обслуживания** (питание, гигиена, уход за собой);
- **утрата способности к самостоятельному передвижению**;
- *